In [ ]:
# Install Mamba SSM (requires CUDA — use Colab with A100 GPU)
!pip install mamba-ssm causal-conv1d -q

In [ ]:
# ── Imports & Setup ─────────────────────────────────────────
import random, os, time, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from mamba_ssm import Mamba as MambaBlock
    MAMBA_AVAILABLE = True
    print('mamba-ssm loaded')
except ImportError:
    MAMBA_AVAILABLE = False
    print('WARNING: mamba-ssm not available — Mamba experiments will be skipped')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Mount Drive & Set Directories ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/word_problem_results/BS12'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Results → {OUTPUT_DIR}')

In [ ]:
# ── Load Datasets ───────────────────────────────────────────
DATA_DIR = '/content/BS12_subset_sum'

df_synthetic = pd.read_csv(os.path.join(DATA_DIR, 'synthetic_100000.csv'))
df_augmented = pd.read_csv(os.path.join(DATA_DIR, 'augmented_100000.csv'))
df_hard      = pd.read_csv(os.path.join(DATA_DIR, 'hard_100000.csv'))

datasets = {
    'synthetic': df_synthetic,
    'augmented': df_augmented,
    'hard':      df_hard,
}

for name, df in datasets.items():
    pos = df['label'].sum()
    neg = len(df) - pos
    print(f'{name:12s}: {len(df):>6} rows | pos={pos} neg={neg} | '
          f'k∈[{df["num_elements"].min()},{df["num_elements"].max()}]')

print()
print(df_synthetic.head())

In [ ]:
# ── Tokenizer & Dataset ─────────────────────────────────────
#
# Input format:  "w1 | w2 | ... | wk || target"
#   - Each wi and the target are space-separated generator tokens: a A b B
#   - |  separates group elements
#   - || separates the element list from the target
#
# Vocabulary:
#   0  PAD
#   1  a          generator
#   2  A          inverse of a
#   3  b          generator
#   4  B          inverse of b
#   5  SEP        element separator  (|)
#   6  TARGET_SEP target separator   (||)
#   7  THINK      scratchpad token   (CoT only)

PAD_ID    = 0
SEP_ID    = 5
TARGET_ID = 6
THINK_ID  = 7

TOKEN_MAP = {'a': 1, 'A': 2, 'b': 3, 'B': 4}

BASE_VOCAB_SIZE = 7   # PAD a A b B SEP TARGET
COT_VOCAB_SIZE  = 8   # + THINK

MAX_SEQ_LEN = 2048


def encode_base(instance_str):
    """Encode instance into base token ids."""
    elems_part, target_part = instance_str.split(' || ', 1)
    elements = elems_part.split(' | ')

    tokens = []
    for i, elem in enumerate(elements):
        if i > 0:
            tokens.append(SEP_ID)
        for ch in elem.split():
            tid = TOKEN_MAP.get(ch)
            if tid is not None:
                tokens.append(tid)

    tokens.append(TARGET_ID)
    for ch in target_part.split():
        tid = TOKEN_MAP.get(ch)
        if tid is not None:
            tokens.append(tid)

    return tokens[:MAX_SEQ_LEN]


def encode_cot(instance_str, think_per_elem=4, think_after_target=8):
    """Encode with THINK scratchpad tokens after each element and target."""
    elems_part, target_part = instance_str.split(' || ', 1)
    elements = elems_part.split(' | ')

    tokens = []
    for i, elem in enumerate(elements):
        if i > 0:
            tokens.append(SEP_ID)
        for ch in elem.split():
            tid = TOKEN_MAP.get(ch)
            if tid is not None:
                tokens.append(tid)
        tokens.extend([THINK_ID] * think_per_elem)

    tokens.append(TARGET_ID)
    for ch in target_part.split():
        tid = TOKEN_MAP.get(ch)
        if tid is not None:
            tokens.append(tid)
    tokens.extend([THINK_ID] * think_after_target)

    return tokens[:MAX_SEQ_LEN]


# Quick sanity check
sample = df_synthetic['instance'].iloc[0]
print(f'Sample:      {sample[:80]}...')
base_enc = encode_base(sample)
cot_enc  = encode_cot(sample)
print(f'Base tokens: {base_enc[:30]}...  (len={len(base_enc)})')
print(f'CoT  tokens: {cot_enc[:30]}...   (len={len(cot_enc)})')


# ── Dataset & Collate ───────────────────────────────────────
class SubsetSumDataset(Dataset):
    def __init__(self, instances, labels, encode_fn):
        self.instances = list(instances)
        self.labels    = list(labels)
        self.encode_fn = encode_fn

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        tokens = self.encode_fn(self.instances[idx])
        return tokens, self.labels[idx], len(tokens)


def collate_fn(batch):
    tokens_list, labels, lengths = zip(*batch)
    max_len = max(lengths)

    padded = [t + [PAD_ID] * (max_len - len(t)) for t in tokens_list]

    # Sort descending by length (needed by pack_padded_sequence for LSTM)
    order = sorted(range(len(lengths)), key=lambda i: lengths[i], reverse=True)

    return (
        torch.tensor([padded[i]  for i in order], dtype=torch.long),
        torch.tensor([labels[i]  for i in order], dtype=torch.float),
        torch.tensor([lengths[i] for i in order], dtype=torch.long),
    )

print('\nTokenizer and dataset classes ready.')

## Model Architectures

We evaluate **four** architectures on the BS(1,2) subset sum decision problem:

| # | Architecture | Key Idea | Params |
|---|-------------|----------|--------|
| 1 | **Bidirectional LSTM** | embed=64, hidden=128, 2 layers, dropout=0.3 | ~330K |
| 2 | **Transformer** | embed=64, 2 layers, 4 heads, ff=256, dropout=0.3 | ~120K |
| 3 | **Transformer + CoT** | Same Transformer, but input augmented with THINK scratchpad tokens | ~120K |
| 4 | **Mamba** | embed=64, 2 layers, state_dim=16, d_conv=4, expand=2, dropout=0.3 | ~105K |

---

### Chain-of-Thought (CoT) for Subset Sum

In language models, chain-of-thought prompting elicits intermediate reasoning steps
before the final answer. For a **classification** task we cannot generate CoT text,
so instead we give the model extra *computation positions*.

**Ideal CoT** for BS(1,2) subset sum would include:
- The Britton normal form $(m_i, k_i, n_i)$ of each element $w_i$
- Trial products for candidate subsets
- Comparison of each trial product with the target

This requires checking exponentially many subsets, reflecting the problem's NP-completeness.

**Our simplified approach:** We insert learnable **THINK tokens** (4 per element,
8 after the target) into the input sequence. During self-attention, the Transformer can
use these positions as a *scratchpad* — storing intermediate representations that combine
information from multiple elements. The THINK token has its own learned embedding;
the model discovers through backpropagation what information to write there.

This is analogous to giving the model working memory for implicit chain-of-thought
computation, following the "thinking tokens" paradigm studied in recent work on
length generalisation in Transformers.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. Bidirectional LSTM
# ═══════════════════════════════════════════════════════════
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids)
        packed = pack_padded_sequence(x, lengths.cpu(),
                                     batch_first=True, enforce_sorted=True)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[-2], h[-1]], dim=1)   # fwd + bwd final hidden
        return self.classifier(h).squeeze(-1)


# ═══════════════════════════════════════════════════════════
# 2. Transformer  (also used for CoT variant with larger vocab)
# ═══════════════════════════════════════════════════════════
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4,
                 num_layers=2, ff_dim=256, dropout=0.3,
                 max_len=MAX_SEQ_LEN):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_emb   = nn.Embedding(max_len, embed_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, input_ids, lengths):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_emb(pos)
        pad_mask = (input_ids == 0)
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        # Mean-pool over non-pad positions
        mask = (~pad_mask).unsqueeze(-1).float()
        pooled = (x * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.classifier(pooled).squeeze(-1)


# ═══════════════════════════════════════════════════════════
# 3. Mamba (Selective State Space Model)
# ═══════════════════════════════════════════════════════════
if MAMBA_AVAILABLE:
    class MambaClassifier(nn.Module):
        def __init__(self, vocab_size, embed_dim=64, num_layers=2,
                     d_state=16, dropout=0.3):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
            self.layers = nn.ModuleList([
                MambaBlock(d_model=embed_dim, d_state=d_state,
                           d_conv=4, expand=2)
                for _ in range(num_layers)
            ])
            self.norms = nn.ModuleList([
                nn.LayerNorm(embed_dim) for _ in range(num_layers)
            ])
            self.drop = nn.Dropout(dropout)
            self.classifier = nn.Sequential(
                nn.Linear(embed_dim, embed_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, 1),
            )

        def forward(self, input_ids, lengths):
            x = self.embedding(input_ids)
            for layer, norm in zip(self.layers, self.norms):
                x = x + self.drop(layer(norm(x)))
            # Mean-pool over non-pad positions
            mask = (input_ids != 0).unsqueeze(-1).float()
            pooled = (x * mask).sum(1) / mask.sum(1).clamp(min=1)
            return self.classifier(pooled).squeeze(-1)


# ═══════════════════════════════════════════════════════════
# Model Registry
# ═══════════════════════════════════════════════════════════
MODEL_CONFIGS = {
    'BiLSTM': {
        'class': BiLSTMClassifier,
        'kwargs': dict(vocab_size=BASE_VOCAB_SIZE, embed_dim=64,
                       hidden_dim=128, num_layers=2, dropout=0.3),
        'encode_fn': encode_base,
    },
    'Transformer': {
        'class': TransformerClassifier,
        'kwargs': dict(vocab_size=BASE_VOCAB_SIZE, embed_dim=64,
                       num_heads=4, num_layers=2, ff_dim=256, dropout=0.3),
        'encode_fn': encode_base,
    },
    'Transformer_CoT': {
        'class': TransformerClassifier,
        'kwargs': dict(vocab_size=COT_VOCAB_SIZE, embed_dim=64,
                       num_heads=4, num_layers=2, ff_dim=256, dropout=0.3),
        'encode_fn': encode_cot,
    },
}

if MAMBA_AVAILABLE:
    MODEL_CONFIGS['Mamba'] = {
        'class': MambaClassifier,
        'kwargs': dict(vocab_size=BASE_VOCAB_SIZE, embed_dim=64,
                       num_layers=2, d_state=16, dropout=0.3),
        'encode_fn': encode_base,
    }

print(f'Models registered: {list(MODEL_CONFIGS.keys())}')
for name, cfg in MODEL_CONFIGS.items():
    m = cfg['class'](**cfg['kwargs'])
    p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:20s}  {p:>8,} params')

In [ ]:
# ── Training ────────────────────────────────────────────────
def train_model(model, train_loader, val_loader,
                epochs=30, patience=5, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    best_val_acc = 0.0
    best_state   = None
    wait = 0

    for epoch in range(epochs):
        # ---- train ----
        model.train()
        correct = total = 0
        for ids, labels, lengths in train_loader:
            ids     = ids.to(device)
            labels  = labels.to(device)
            lengths = lengths.to(device)

            optimizer.zero_grad()
            logits = model(ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            preds = (logits > 0).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

        train_acc = correct / total

        # ---- validate ----
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for ids, labels, lengths in val_loader:
                ids     = ids.to(device)
                labels  = labels.to(device)
                lengths = lengths.to(device)
                logits  = model(ids, lengths)
                preds   = (logits > 0).float()
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_acc = correct / total

        # ---- early stopping ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return best_val_acc


# ── Evaluation ───────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for ids, labels, lengths in loader:
            ids     = ids.to(device)
            labels  = labels.to(device)
            lengths = lengths.to(device)
            preds   = (model(ids, lengths) > 0).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


# ── 5-Fold Cross-Validation Runner ──────────────────────────
def run_cv(df, model_name, seed, dataset_name, n_folds=5,
           batch_size=64, epochs=30, patience=5, lr=1e-3):
    """Run stratified 5-fold CV.  Returns list of per-fold result dicts."""
    cfg       = MODEL_CONFIGS[model_name]
    encode_fn = cfg['encode_fn']

    X = df['instance'].values
    y = df['label'].values

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_results = []

    for fold, (train_val_idx, test_idx) in enumerate(skf.split(X, y)):
        # Sub-split: 85 % train, 15 % val (of the training fold)
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.15, random_state=seed,
            stratify=y[train_val_idx],
        )

        train_ds = SubsetSumDataset(X[train_idx], y[train_idx], encode_fn)
        val_ds   = SubsetSumDataset(X[val_idx],   y[val_idx],   encode_fn)
        test_ds  = SubsetSumDataset(X[test_idx],  y[test_idx],  encode_fn)

        train_ld = DataLoader(train_ds, batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=2,
                              pin_memory=True)
        val_ld   = DataLoader(val_ds,   batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2,
                              pin_memory=True)
        test_ld  = DataLoader(test_ds,  batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2,
                              pin_memory=True)

        # Reproducibility
        torch.manual_seed(seed + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed + fold)
        np.random.seed(seed + fold)

        model   = cfg['class'](**cfg['kwargs']).to(device)
        t0      = time.time()
        val_acc = train_model(model, train_ld, val_ld, epochs, patience, lr)
        elapsed = time.time() - t0

        test_acc = evaluate(model, test_ld)

        fold_results.append({
            'model':      model_name,
            'dataset':    dataset_name,
            'seed':       seed,
            'fold':       fold,
            'val_acc':    round(val_acc,  4),
            'test_acc':   round(test_acc, 4),
            'train_time': round(elapsed,  1),
        })

        print(f'  {model_name:20s} | {dataset_name:10s} | '
              f'seed={seed} fold={fold} | '
              f'val={val_acc:.4f}  test={test_acc:.4f}  ({elapsed:.0f}s)')

    return fold_results

print('Training, evaluation, and CV functions ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Main Experiment Loop
#   4 architectures × 3 datasets × 3 seeds × 5 folds = 180 runs
# ═══════════════════════════════════════════════════════════

SEEDS         = [42, 123, 456]
DATASET_NAMES = ['synthetic', 'augmented', 'hard']

all_results = []
n_models = len(MODEL_CONFIGS)
total_runs = n_models * len(DATASET_NAMES) * len(SEEDS) * 5
run_count  = 0

print(f'Total planned runs: {total_runs}')
print(f'Models: {list(MODEL_CONFIGS.keys())}')
print(f'Datasets: {DATASET_NAMES}')
print(f'Seeds: {SEEDS}\n')

t_start = time.time()

for dataset_name in DATASET_NAMES:
    df = datasets[dataset_name]
    for model_name in MODEL_CONFIGS:
        for seed in SEEDS:
            print(f'\n{"="*60}')
            print(f'{model_name} | {dataset_name} | seed={seed}')
            print(f'{"="*60}')

            results = run_cv(df, model_name, seed, dataset_name)
            all_results.extend(results)
            run_count += 5

            # Save partial results after each (model, dataset, seed) combo
            pd.DataFrame(all_results).to_csv(
                f'{OUTPUT_DIR}/BS12_results_partial.csv', index=False
            )

            elapsed = time.time() - t_start
            print(f'  >>> Progress: {run_count}/{total_runs} runs  '
                  f'({elapsed/60:.1f} min elapsed)')

total_time = time.time() - t_start
print(f'\n{"="*60}')
print(f'ALL {run_count} RUNS COMPLETE  ({total_time/60:.1f} min total)')
print(f'{"="*60}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Scaling Test: train on k ≤ 8, test on k ≥ 9
#   Train on synthetic (k=5-8)
#   Test on augmented (k≥9) + hard (k=12-15)
# ═══════════════════════════════════════════════════════════

scale_test_df = pd.concat([
    df_augmented[df_augmented['num_elements'] >= 9],
    df_hard,
]).reset_index(drop=True)

print(f'Scaling train set: {len(df_synthetic)} instances  (k=5-8)')
print(f'Scaling test set:  {len(scale_test_df)} instances  '
      f'(k ∈ [{scale_test_df["num_elements"].min()}, '
      f'{scale_test_df["num_elements"].max()}])')
print(f'Test k distribution:')
print(scale_test_df['num_elements'].value_counts().sort_index())
print()

scaling_results = []
train_df = df_synthetic

for model_name in MODEL_CONFIGS:
    cfg       = MODEL_CONFIGS[model_name]
    encode_fn = cfg['encode_fn']

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)

        # Train/val split on synthetic
        tr_df, va_df = train_test_split(
            train_df, test_size=0.15, random_state=seed,
            stratify=train_df['label'],
        )

        tr_ds = SubsetSumDataset(tr_df['instance'].values,
                                 tr_df['label'].values, encode_fn)
        va_ds = SubsetSumDataset(va_df['instance'].values,
                                 va_df['label'].values, encode_fn)

        tr_ld = DataLoader(tr_ds, 64, shuffle=True,
                           collate_fn=collate_fn, num_workers=2,
                           pin_memory=True)
        va_ld = DataLoader(va_ds, 64, shuffle=False,
                           collate_fn=collate_fn, num_workers=2,
                           pin_memory=True)

        model = cfg['class'](**cfg['kwargs']).to(device)
        train_model(model, tr_ld, va_ld)

        # Evaluate per k-value
        for k_val in sorted(scale_test_df['num_elements'].unique()):
            sub = scale_test_df[scale_test_df['num_elements'] == k_val]
            if len(sub) < 20:
                continue
            test_ds = SubsetSumDataset(sub['instance'].values,
                                       sub['label'].values, encode_fn)
            test_ld = DataLoader(test_ds, 64, shuffle=False,
                                 collate_fn=collate_fn)
            acc = evaluate(model, test_ld)

            scaling_results.append({
                'model': model_name, 'seed': seed,
                'k': k_val, 'test_acc': round(acc, 4), 'n': len(sub),
            })

        print(f'  {model_name:20s}  seed={seed}  scaling done')

scaling_df = pd.DataFrame(scaling_results)
scaling_df.to_csv(f'{OUTPUT_DIR}/BS12_scaling.csv', index=False)

print('\nScaling accuracy by (model, k):')
print(scaling_df.groupby(['model', 'k'])['test_acc'].mean()
      .unstack().round(3))

In [ ]:
# ═══════════════════════════════════════════════════════════
# Save Final Results
# ═══════════════════════════════════════════════════════════

results_df = pd.DataFrame(all_results)
results_df.to_csv(f'{OUTPUT_DIR}/BS12_results.csv', index=False)

# Summary: mean ± std over (3 seeds × 5 folds = 15 runs) per cell
summary = results_df.groupby(['model', 'dataset']).agg(
    mean_acc  = ('test_acc',   'mean'),
    std_acc   = ('test_acc',   'std'),
    mean_val  = ('val_acc',    'mean'),
    mean_time = ('train_time', 'mean'),
).round(4)

summary.to_csv(f'{OUTPUT_DIR}/BS12_summary.csv')

print('=' * 60)
print('SUMMARY — Test accuracy  (mean ± std over 3 seeds × 5 folds)')
print('=' * 60)
for (model, ds), row in summary.iterrows():
    print(f'  {model:20s} | {ds:12s} | '
          f'{row["mean_acc"]:.4f} ± {row["std_acc"]:.4f}  '
          f'({row["mean_time"]:.0f}s avg)')

# ═══════════════════════════════════════════════════════════
# Visualization
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Bar chart: accuracy per dataset
pivot = results_df.pivot_table('test_acc', 'dataset', 'model', 'mean')
pivot.plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('BS(1,2) Subset Sum — Test Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.4, 1.05)
axes[0].legend(fontsize=7, loc='lower left')

# 2. Box plot: accuracy distribution across folds/seeds
results_df.boxplot(column='test_acc', by='model', ax=axes[1])
axes[1].set_title('Accuracy Distribution (all datasets)')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_xlabel('')
plt.sca(axes[1])
plt.xticks(rotation=25, ha='right', fontsize=8)

# 3. Scaling curve: accuracy vs k
if len(scaling_df) > 0:
    for mname in sorted(scaling_df['model'].unique()):
        sub = scaling_df[scaling_df['model'] == mname]
        agg = sub.groupby('k')['test_acc'].mean()
        axes[2].plot(agg.index, agg.values, 'o-', label=mname, markersize=4)
    axes[2].set_title('Scaling: Accuracy vs k  (trained on k≤8)')
    axes[2].set_xlabel('Number of elements (k)')
    axes[2].set_ylabel('Test Accuracy')
    axes[2].legend(fontsize=7)
    axes[2].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    axes[2].set_ylim(0.3, 1.05)

plt.suptitle('')  # remove auto suptitle from boxplot
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/BS12_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nAll results saved to {OUTPUT_DIR}/')
print(f'  BS12_results.csv   — {len(results_df)} rows (all fold results)')
print(f'  BS12_summary.csv   — aggregated mean ± std')
print(f'  BS12_scaling.csv   — {len(scaling_df)} rows (scaling test)')
print(f'  BS12_plots.png     — visualizations')